In [ ]:
import sys

# 05a_construir_lexicon

## 1. Objetivo
Construir `lexicons/processed/hatecr_lexicon.csv` a partir de fuentes externas colocadas manualmente en `lexicons/raw/`, usando un flujo reproducible de limpieza, lematizacion y estandarizacion.


## 2. Fuentes esperadas
- Archivos colocados manualmente en `lexicons/raw/` (`.csv`, `.tsv`, `.txt`, `.xlsx`).
- Columnas minimas recomendadas por fuente: termino y, si existe, categoria.
- No se descargan recursos automaticamente en este notebook.


## 3. Carga de archivos desde lexicons/raw/


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display




def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config").exists() and (candidate / "src").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.lexicon import list_raw_lexicon_files, load_raw_lexicon_files

RAW_DIR = PROJECT_ROOT / "lexicons" / "raw"
raw_files = list_raw_lexicon_files(RAW_DIR)

if not raw_files:
    raise FileNotFoundError(
        f"No hay archivos en {RAW_DIR}. Coloca las fuentes manualmente y vuelve a ejecutar."
    )

raw_sources = load_raw_lexicon_files(RAW_DIR)

print(f"Project root: {PROJECT_ROOT}")
print(f"Archivos detectados en raw: {len(raw_files)}")
for path in raw_files:
    print(f"- {path.name}")

for source_name, source_df in raw_sources.items():
    print(f"\n[{source_name}] shape={source_df.shape}")
    display(source_df.head(3))


In [ ]:
from src.lexicon import standardize_raw_lexicon

SOURCE_SCHEMAS = {
    "Hate Speech Library in Spain": {
        "term": "Lemas",
    },

    "hurtlex_ES": {
        "term": "lemma",
    },

    "immigrant_lexicon": {
        "term": "termino",
    },

    "insults_lexicon": {
        "term": "termino",
    },

    "misogyny_lexicon": {
        "term": "termino",
    },

    "xenophobia_lexicon": {
        "term": "termino",
    },
}

print(SOURCE_SCHEMAS)


## 4. Normalización de texto


In [ ]:


normalized_sources = {}
for source_name, source_df in raw_sources.items():
    schema = SOURCE_SCHEMAS.get(source_name, {})
    normalized_sources[source_name] = standardize_raw_lexicon(
        source_df,
        source=source_name,
        schema=schema,
        default_category="uncategorized",
        default_target_type="unknown",
        default_severity=1.0,
        default_context_dependency="medium",
        include_in_queries_default=False,
        include_in_classification_default=True,
    )

for source_name, source_df in normalized_sources.items():
    print(f"[{source_name}] filas normalizadas: {len(source_df)}")
    display(source_df.head(3))


## 5. Lematización en español con spaCy


In [ ]:
from src.lexicon import load_spacy_spanish_model, lemmatize_terms

try:
    nlp_es = load_spacy_spanish_model()
    print("Modelo spaCy en espanol cargado correctamente.")
except OSError as exc:
    nlp_es = None
    print("No se pudo cargar un modelo de spaCy para espanol.")
    print(exc)
    print("Continuando con lemma=term hasta que instales el modelo manualmente.")

lemmatized_sources = {}
for source_name, source_df in normalized_sources.items():
    if nlp_es is None:
        tmp_df = source_df.copy()
        tmp_df["lemma"] = tmp_df["term"]
        lemmatized_sources[source_name] = tmp_df
    else:
        lemmatized_sources[source_name] = lemmatize_terms(source_df, nlp=nlp_es)

for source_name, source_df in lemmatized_sources.items():
    print(f"[{source_name}] filas lematizadas: {len(source_df)}")
    display(source_df[["term", "lemma", "category", "source"]].head(5))


## 6. Unión de fuentes


In [ ]:
from src.lexicon import union_lexicon_sources

lexicon_union = union_lexicon_sources(lemmatized_sources.values())
print(f"Filas despues de unir fuentes: {len(lexicon_union)}")
display(lexicon_union.head(10))


## 7. Eliminación de duplicados


In [ ]:
from src.lexicon import deduplicate_lexicon

before_dedup = len(lexicon_union)
lexicon_dedup = deduplicate_lexicon(
    lexicon_union,
    subset=["term", "lemma", "category", "source"],
    keep="first",
)
after_dedup = len(lexicon_dedup)

print(f"Antes de deduplicar: {before_dedup}")
print(f"Despues de deduplicar: {after_dedup}")
print(f"Duplicados eliminados: {before_dedup - after_dedup}")
display(lexicon_dedup.head(10))


## 8. Asignación de categorías


In [ ]:
from src.lexicon import assign_categories, ensure_final_schema

# Mapeo opcional de categorias originales a categorias estandar HateCR.
# Completar manualmente segun criterio metodologico.
CATEGORY_MAP = {
    # "categoria_original": "categoria_estandar"
}

# Opcional: perfiles por categoria para completar metadatos.
CATEGORY_PROFILES = {
    # "categoria_estandar": {
    #     "target_type": "group_or_individual",
    #     "severity": 1.0,
    #     "context_dependency": "medium",
    #     "include_in_queries": False,
    #     "include_in_classification": True,
    # }
}

lexicon_categorized = assign_categories(
    lexicon_dedup,
    category_map=CATEGORY_MAP,
    default_category="uncategorized",
)

for category_name, profile in CATEGORY_PROFILES.items():
    mask = lexicon_categorized["category"] == category_name
    for field, value in profile.items():
        lexicon_categorized.loc[mask, field] = value

lexicon_final = ensure_final_schema(lexicon_categorized)

print("Distribucion de categorias:")
print(lexicon_final["category"].value_counts(dropna=False))
display(lexicon_final.head(10))


## 9. Exportación del lexicón procesado


In [ ]:
from src.lexicon import FINAL_LEXICON_COLUMNS, export_processed_lexicon

OUTPUT_PATH = PROJECT_ROOT / "lexicons" / "processed" / "hatecr_lexicon.csv"
exported_path = export_processed_lexicon(lexicon_final, OUTPUT_PATH)

exported_df = pd.read_csv(exported_path)
print(f"Lexicon exportado en: {exported_path}")
print(f"Filas exportadas: {len(exported_df)}")
print("Columnas exportadas:")
print(exported_df.columns.tolist())

if exported_df.columns.tolist() != FINAL_LEXICON_COLUMNS:
    raise ValueError(
        "Las columnas del lexicon exportado no coinciden con el contrato esperado."
    )

display(exported_df)

## 10. Advertencia metodológica
- Un lexicon no sustituye la interpretacion contextual ni la codificacion manual experta.
- Pueden existir falsos positivos y falsos negativos por ironia, polisemia o citacion.
- Cualquier uso del lexicon para clasificacion debe validarse con muestras anotadas manualmente.
- Documenta fuentes, licencias y decisiones de curaduria en cada iteracion.
